# Iteration 2 – monthly request trends by Council District

**Owner:** Akshara Annu · **Reviewer:** Ma'el Hashemee · **Task:** 1 in `docs/tasks/iter2-akshara.md`

This notebook counts GlendaleOne requests per Council District per month and produces every number used in `outputs/iter2/trend/`:

| Output | What it holds |
|---|---|
| `monthly_counts_by_district.csv` | One row per district-month: `district`, `month`, `n` |
| `change_by_district.csv` | 2023 vs 2024 totals, January to July 2025 vs 2026 totals, the highest and lowest month, per district plus a citywide row |
| `monthly_trend_by_district.png` | Six small-multiple panels on one shared y-axis |
| `trend_charts.html` | Akshara's page; this notebook rewrites its data block and fills its marked text spots |

`findings.md` in the same folder is written by hand from the printed results below.

- **Source:** `data/cleaned/glendaleone_clean.csv` (City of Glendale GlendaleOne External Requests, extract loaded 2026-08-06).
- **Window:** `request_date` from 2020-01-01 to 2026-07-31 (79 full months).
- **Grain:** one input row is one request; one output row is one district-month.
- **Denominator:** none. These are raw counts, not requests per resident.
- The analysis describes patterns only. It does not test or claim reasons for them.

## 1. Load and filter

The standard load block from `docs/ANALYSIS_STANDARDS.md` section 2, unchanged. It prints `n_raw`, `n_window`, `n_final` and the number of in-window rows without a district.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "cleaned"
DISTRICTS = ["BARREL", "CACTUS", "CHOLLA", "OCOTILLO", "SAHUARO", "YUCCA"]
START, END = "2020-01-01", "2026-07-31"      # 79 full months
YEARS = 79 / 12

req = pd.read_csv(DATA / "glendaleone_clean.csv",
                  parse_dates=["request_date", "close_date"], low_memory=False)
req["district"] = req["district"].astype("string").str.strip().str.upper()
n_raw = len(req)
req_w = req[req["request_date"].between(START, END)]
n_window = len(req_w)
req_w = req_w[req_w["district"].isin(DISTRICTS)].copy()
n_final = len(req_w)
print(n_raw, n_window, n_final, n_window - n_final)   # last number = rows without a district

## 2. Input checks

These checks stop the notebook if the input differs from what `docs/ANALYSIS_STANDARDS.md` describes:

- the extract date in `date_loaded` is 2026-08-06 for every row;
- `n_raw` is 106,795, and the rows outside the window (before 2020-01-01, after 2026-07-31, or with no date) add up to `n_raw - n_window`;
- `request_date` is a datetime, `district` is a string, and only the six district names remain;
- `request_number` is unique and every kept date falls inside the window;
- null counts for the two columns this analysis uses.

`OUT` is the only folder the notebook writes to.

In [ ]:
import json
import re
from html import escape

OUT = ROOT / "outputs" / "iter2" / "trend"
OUT.mkdir(parents=True, exist_ok=True)

extract_dates = sorted(pd.to_datetime(req["date_loaded"]).dt.strftime("%Y-%m-%d").unique())
EXTRACT = extract_dates[0]
n_before = int((req["request_date"] < pd.Timestamp(START)).sum())
n_after = int((req["request_date"] > pd.Timestamp(END)).sum())
n_no_date = int(req["request_date"].isna().sum())
n_no_district = n_window - n_final

assert extract_dates == ["2026-08-06"]
assert n_raw == 106795
assert n_before + n_after + n_no_date == n_raw - n_window
assert pd.api.types.is_datetime64_any_dtype(req_w["request_date"])
assert pd.api.types.is_string_dtype(req_w["district"])
assert sorted(req_w["district"].unique()) == DISTRICTS
assert req_w["request_number"].is_unique
assert req_w["request_date"].min() >= pd.Timestamp(START)
assert req_w["request_date"].max() <= pd.Timestamp(END)

print("extract loaded:", EXTRACT)
print(f"n_raw {n_raw:,} | before window {n_before:,} | after window {n_after:,} | no date {n_no_date:,}")
print(f"n_window {n_window:,} | no district {n_no_district:,} ({n_no_district / n_window:.1%}) | n_final {n_final:,}")
print("nulls in the window:", req[req["request_date"].between(START, END)][["request_date", "district"]].isna().sum().to_dict())

## 3. Monthly counts by district

Each request is assigned to the calendar month of its `request_date`. The counts sit on the full grid of 6 districts x 79 months, so a month with no requests would appear as a 0 and fail the check.

Checks:

- 6 x 79 = 474 district-months;
- no empty district-month (every `n` is above 0);
- the counts sum to `n_final`;
- per-district totals agree when counted a second way (`value_counts` on the filtered rows instead of summing the monthly table).

The table is written to `monthly_counts_by_district.csv` and read back, so every step after this one uses the file on disk.

In [ ]:
months = pd.period_range(START, END, freq="M").astype(str)
grid = pd.MultiIndex.from_product([DISTRICTS, months], names=["district", "month"])
monthly = (
    req_w.assign(month=req_w["request_date"].dt.to_period("M").astype(str))
    .groupby(["district", "month"])
    .size()
    .reindex(grid, fill_value=0)
    .rename("n")
    .reset_index()
)
monthly["district"] = monthly["district"].astype(str)

n_months = len(months)
n_district_months = len(monthly)
totals_sum = monthly.groupby("district")["n"].sum().reindex(DISTRICTS)
totals_count = req_w["district"].value_counts().reindex(DISTRICTS)

assert n_months == 79
assert n_district_months == len(DISTRICTS) * n_months == 474
assert (monthly["n"] > 0).all()
assert int(monthly["n"].sum()) == n_final
assert (totals_sum.to_numpy() == totals_count.to_numpy()).all()

monthly.to_csv(OUT / "monthly_counts_by_district.csv", index=False)
monthly_csv = pd.read_csv(OUT / "monthly_counts_by_district.csv")
pd.testing.assert_frame_equal(monthly_csv, monthly)

print(f"{n_district_months} district-months, {n_months} months, sum {int(monthly_csv['n'].sum()):,} = n_final {n_final:,}")
print(f"smallest district-month {monthly_csv['n'].min()}, largest {monthly_csv['n'].max()}")
pd.DataFrame({"sum_of_monthly_n": totals_sum, "value_counts": totals_count})

## 4. Year-over-year and January to July changes

`change_by_district.csv` has one row per district plus a `CITYWIDE` row (the sum of the six districts, month by month). All values come from `monthly_counts_by_district.csv`.

- `n_2023`, `n_2024`: calendar-year totals. Both years are complete in the window.
- `n_jan_jul_2025`, `n_jan_jul_2026`: January to July totals. The window ends on 2026-07-31, so 2026 is compared only with the same seven months of 2025.
- Percent change = (newer period / older period - 1) x 100, rounded to one decimal.
- `peak_month`, `peak_n`, `low_month`, `low_n`: the highest and lowest month in the 79-month window.

In [ ]:
work = monthly_csv.assign(year=monthly_csv["month"].str[:4].astype(int),
                          mon=monthly_csv["month"].str[5:7].astype(int))
citywide = (work.groupby(["month", "year", "mon"], as_index=False)["n"].sum()
            .assign(district="CITYWIDE"))
both = pd.concat([work, citywide[work.columns]], ignore_index=True)


def pct(new, old):
    return round((new / old - 1) * 100, 1)


records = []
for name, g in both.groupby("district", sort=False):
    jan_jul = g[g["mon"] <= 7]
    n_2023 = int(g.loc[g["year"] == 2023, "n"].sum())
    n_2024 = int(g.loc[g["year"] == 2024, "n"].sum())
    n_jj_2025 = int(jan_jul.loc[jan_jul["year"] == 2025, "n"].sum())
    n_jj_2026 = int(jan_jul.loc[jan_jul["year"] == 2026, "n"].sum())
    peak = g.loc[g["n"].idxmax()]
    low = g.loc[g["n"].idxmin()]
    records.append({
        "district": name,
        "n_total": int(g["n"].sum()),
        "n_2023": n_2023,
        "n_2024": n_2024,
        "pct_change_2023_2024": pct(n_2024, n_2023),
        "n_jan_jul_2025": n_jj_2025,
        "n_jan_jul_2026": n_jj_2026,
        "pct_change_jan_jul_2025_2026": pct(n_jj_2026, n_jj_2025),
        "peak_month": peak["month"],
        "peak_n": int(peak["n"]),
        "low_month": low["month"],
        "low_n": int(low["n"]),
    })
change = pd.DataFrame(records)

parts = change[change["district"] != "CITYWIDE"]
city_row = change[change["district"] == "CITYWIDE"].iloc[0]
assert list(parts["district"]) == DISTRICTS
assert len(citywide) == n_months
for col in ["n_total", "n_2023", "n_2024", "n_jan_jul_2025", "n_jan_jul_2026"]:
    assert int(parts[col].sum()) == int(city_row[col])
assert int(city_row["n_total"]) == n_final

change.to_csv(OUT / "change_by_district.csv", index=False)
change_csv = pd.read_csv(OUT / "change_by_district.csv")
change_csv

## 5. Headline numbers

Every number in `findings.md`, the chart title and the page is printed here. The checks guard the wording: for example, the page says "all six districts" fell from 2023 to 2024 only if every district's change is below zero, and the highest district-month and highest and lowest citywide months must each be a single month (no ties).

In [ ]:
dist = change_csv[change_csv["district"] != "CITYWIDE"].set_index("district")
city = change_csv[change_csv["district"] == "CITYWIDE"].iloc[0]
city_months = citywide.set_index("month")["n"]
top_dm = monthly_csv.loc[monthly_csv["n"].idxmax()]
low_dm = monthly_csv.loc[monthly_csv["n"].idxmin()]
largest_drop = dist["pct_change_2023_2024"].idxmin()
smallest_drop = dist["pct_change_2023_2024"].idxmax()
jj_order = dist["pct_change_jan_jul_2025_2026"].sort_values(ascending=False, kind="stable")
nov_2024 = int(city_months["2024-11"])
nov_2024_rank = int(city_months.rank(method="min")["2024-11"])
nov_2024_district_lows = list(dist.index[dist["low_month"] == "2024-11"])

assert (dist["pct_change_2023_2024"] < 0).all()
assert int((monthly_csv["n"] == top_dm["n"]).sum()) == 1
assert int((monthly_csv["n"] == low_dm["n"]).sum()) == 1
assert int((city_months == city_months.max()).sum()) == 1
assert int((city_months == city_months.min()).sum()) == 1
assert city["low_month"] == "2024-11" and nov_2024_rank == 1


def fmt_pct(value):
    return f"{value:+.1f}%"


def month_label(month, style="%B %Y"):
    return pd.Period(month, freq="M").strftime(style)


cactus = work[work["district"] == "CACTUS"].set_index("month")["n"]

print("2023 to 2024 by district:")
print(dist[["n_2023", "n_2024", "pct_change_2023_2024"]].sort_values("pct_change_2023_2024").to_string())
print(f"citywide: {city['n_2023']:,} to {city['n_2024']:,} ({fmt_pct(city['pct_change_2023_2024'])})")
print(f"all six districts fell: {bool((dist['pct_change_2023_2024'] < 0).all())}; "
      f"largest drop {largest_drop}, smallest drop {smallest_drop}")
print()
print(f"highest district-month: {top_dm['district']} {top_dm['month']} = {top_dm['n']}")
print(f"lowest district-month: {low_dm['district']} {low_dm['month']} = {low_dm['n']}")
print(f"highest citywide month: {city['peak_month']} = {city['peak_n']:,}")
print(f"lowest citywide month: {city['low_month']} = {city['low_n']:,}")
print()
print("January to July, 2026 vs 2025:")
print(dist[["n_jan_jul_2025", "n_jan_jul_2026", "pct_change_jan_jul_2025_2026"]]
      .loc[jj_order.index].to_string())
print(f"citywide: {city['n_jan_jul_2025']:,} to {city['n_jan_jul_2026']:,} ({fmt_pct(city['pct_change_jan_jul_2025_2026'])})")
print()
print("Cactus, March to July 2023:", cactus.loc["2023-03":"2023-07"].to_dict())
print("Cactus, second-highest month:", cactus.drop(top_dm["month"]).idxmax(), cactus.drop(top_dm["month"]).max())
print(f"Cactus median month: {cactus.median():.0f}")
print()
print(f"November 2024 citywide: {nov_2024:,}, rank {nov_2024_rank} of {n_months} months (1 = lowest)")
print("five lowest citywide months:", city_months.nsmallest(5).to_dict())
print("districts whose own lowest month is November 2024:", nov_2024_district_lows)
print("district values in November 2024:", work[work["month"] == "2024-11"].set_index("district")["n"].to_dict())

## 6. Six-panel chart

The chart-standard block from `docs/ANALYSIS_STANDARDS.md` section 7, unchanged, followed by the chart. One panel per district, one hue (`BLUE`), the same y-axis on every panel starting at zero with round ticks, and labels only on the two extremes (the highest and the lowest district-month). The title is built from the numbers above.

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="talk")        # seaborn 0.13.2
BLUE = "#2a78d6"                                        # default single hue
DISTRICT_COLORS = {"BARREL": "#2a78d6", "CACTUS": "#eb6834", "CHOLLA": "#1baf7a",
                   "OCOTILLO": "#eda100", "SAHUARO": "#e87ba4", "YUCCA": "#008300"}

In [ ]:
import matplotlib.dates as mdates

Y_STEP = 100
y_max = int((monthly_csv["n"].max() // Y_STEP + 1) * Y_STEP)
y_ticks = list(range(0, y_max + 1, Y_STEP))

chart_title = (f"Requests fell in all six districts from 2023 to 2024: "
               f"{largest_drop.title()} {fmt_pct(dist.loc[largest_drop, 'pct_change_2023_2024'])}, "
               f"citywide {fmt_pct(city['pct_change_2023_2024'])}")
chart_subtitle = (f"Monthly GlendaleOne requests by Council District, {month_label(months[0])} to "
                  f"{month_label(months[-1])} ({n_months} months). Raw counts, not per resident; "
                  f"same 0 to {y_max} scale on every panel.")
source_note = f"Source: City of Glendale GlendaleOne External Requests (extract loaded {EXTRACT})."

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 6.75), sharex=True, sharey=True)
for ax, name in zip(axes.flat, DISTRICTS):
    g = monthly_csv[monthly_csv["district"] == name]
    ax.plot(pd.to_datetime(g["month"]), g["n"], color=BLUE, linewidth=1.6)
    ax.set_title(name.title(), loc="left", fontsize=14, fontweight="bold", pad=6)
    ax.set_ylim(0, y_max)
    ax.set_yticks(y_ticks)
    ax.xaxis.set_major_locator(mdates.YearLocator(base=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="both", labelsize=11, length=0, pad=4)
    ax.grid(axis="x", visible=False)
    ax.grid(axis="y", color="#e3e3e3", linewidth=0.8)
    ax.set_xlabel("")
    ax.set_ylabel("")
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("#bbbbbb")
    ax.spines["bottom"].set_linewidth(0.8)
for ax in axes[:, 0]:
    ax.set_ylabel("Requests per month", fontsize=12)

for mark, dx, dy, ha, va in [(top_dm, 7, 0, "left", "center"), (low_dm, -7, -4, "right", "top")]:
    ax = axes.flat[DISTRICTS.index(mark["district"])]
    mx = mdates.date2num(pd.Timestamp(mark["month"]))
    ax.scatter([mx], [mark["n"]], color=BLUE, s=28, zorder=3)
    ax.annotate(f"{mark['n']} in {month_label(mark['month'], '%b %Y')}", xy=(mx, mark["n"]),
                xytext=(dx, dy), textcoords="offset points", fontsize=11, ha=ha, va=va, color="#222222")

fig.subplots_adjust(left=0.08, right=0.99, top=0.80, bottom=0.10, wspace=0.06, hspace=0.30)
fig.text(0.01, 0.975, chart_title, fontsize=16, fontweight="bold", ha="left", va="top")
fig.text(0.01, 0.91, chart_subtitle, fontsize=11.5, color="#444444", ha="left", va="top")
fig.text(0.01, 0.005, source_note, fontsize=10, color="#555555", ha="left", va="bottom")
fig.savefig(OUT / "monthly_trend_by_district.png", dpi=200, bbox_inches="tight")
print(chart_title)
print(chart_subtitle)
print("y ticks:", y_ticks)
plt.show()

## 7. HTML page

`trend_charts.html` keeps Akshara's design. The notebook rebuilds the block between the `/*fill:data*/` markers: `rows` comes straight from `monthly_counts_by_district.csv` (read back from disk) and `meta` holds the y-axis scale, the two labelled extremes and one note per district from `change_by_district.csv`. Each text spot marked `<!--fill:key-->` gets its sentence or number from the values above. Running the notebook again rewrites the same spots, so the page always matches the CSVs.

In [ ]:
page_path = OUT / "trend_charts.html"
page = page_path.read_text(encoding="utf-8")

rows_js = json.loads(pd.read_csv(OUT / "monthly_counts_by_district.csv").to_json(orient="records"))
panels = {
    name: {
        "name": name.title(),
        "note": (f"2023 to 2024: {fmt_pct(r['pct_change_2023_2024'])} · "
                 f"Jan to Jul, 2026 vs 2025: {fmt_pct(r['pct_change_jan_jul_2025_2026'])} · "
                 f"highest month: {r['peak_n']} ({month_label(r['peak_month'], '%b %Y')})"),
    }
    for name, r in dist.iterrows()
}
meta = {
    "y_max": y_max,
    "y_ticks": y_ticks,
    "panels": panels,
    "peak": {"district": top_dm["district"], "month": top_dm["month"], "n": int(top_dm["n"]),
             "label": f"{top_dm['n']} in {month_label(top_dm['month'], '%b %Y')}"},
    "low": {"district": low_dm["district"], "month": low_dm["month"], "n": int(low_dm["n"]),
            "label": f"{low_dm['n']} in {month_label(low_dm['month'], '%b %Y')}"},
}
data_js = f"\nconst rows = {json.dumps(rows_js)};\nconst meta = {json.dumps(meta)};\n"

rises = [n for n in jj_order.index if jj_order[n] > 0]
falls = [n for n in jj_order.index if jj_order[n] < 0]


def listing(names):
    items = [f"{n.title()} {fmt_pct(jj_order[n])}" for n in names]
    return ", ".join(items[:-1]) + " and " + items[-1] if len(items) > 1 else items[0]


top_up, top_down = jj_order.index[0], jj_order.index[-1]
fills = {
    "title": (f"All six districts filed fewer requests in 2024 than in 2023; "
              f"{largest_drop.title()} fell the most ({fmt_pct(dist.loc[largest_drop, 'pct_change_2023_2024'])})"),
    "lead": (f"{month_label(months[0])} to {month_label(months[-1])} ({n_months} months) · monthly GlendaleOne "
             f"request counts by Council District · raw counts, not per resident · every panel uses the "
             f"same 0 to {y_max} request scale."),
    "obs_1": (f"Every district filed fewer requests in 2024 than in 2023. {largest_drop.title()} fell the most, "
              f"{fmt_pct(dist.loc[largest_drop, 'pct_change_2023_2024'])} ({dist.loc[largest_drop, 'n_2023']:,} to "
              f"{dist.loc[largest_drop, 'n_2024']:,}), and {smallest_drop.title()} the least, "
              f"{fmt_pct(dist.loc[smallest_drop, 'pct_change_2023_2024'])}. Citywide: "
              f"{fmt_pct(city['pct_change_2023_2024'])} ({city['n_2023']:,} to {city['n_2024']:,})."),
    "obs_2": (f"The highest single district-month was {top_dm['district'].title()} in "
              f"{month_label(top_dm['month'])}, with {top_dm['n']} requests. The highest citywide month was "
              f"{month_label(city['peak_month'])}, with {city['peak_n']:,} requests across the six districts."),
    "obs_3": (f"January to July 2026 against the same months of 2025: {listing(rises)} rose; "
              f"{listing(falls)} fell. Citywide: {fmt_pct(city['pct_change_jan_jul_2025_2026'])}."),
    "obs_4": (f"Citywide, {month_label(city['low_month'])} had {city['low_n']:,} requests, the lowest of all "
              f"{n_months} months in the window."),
    "kpi_window": f"{n_months} months",
    "kpi_window_note": f"{month_label(months[0], '%b %Y')} to {month_label(months[-1], '%b %Y')}",
    "kpi_districts": f"{len(DISTRICTS)}",
    "kpi_districts_note": f"{n_final:,} requests in {n_district_months} district-months",
    "kpi_peak": f"{top_dm['n']}",
    "kpi_peak_note": f"{top_dm['district'].title()}, {month_label(top_dm['month'])}",
    "kpi_drop": fmt_pct(dist.loc[largest_drop, "pct_change_2023_2024"]),
    "kpi_drop_note": f"{largest_drop.title()} (citywide {fmt_pct(city['pct_change_2023_2024'])})",
    "section_title": (f"In 2026 the districts moved in different directions: {top_up.title()} "
                      f"{fmt_pct(jj_order[top_up])}, {top_down.title()} {fmt_pct(jj_order[top_down])} "
                      f"(January to July, 2026 vs 2025)"),
    "footer_source": f"City of Glendale GlendaleOne External Requests, cleaned team extract loaded {EXTRACT}.",
    "footer_filters": (f"Requests outside {START} through {END} ({n_raw - n_window:,}) and requests without one of "
                       f"the six Council Districts ({n_no_district:,}) are excluded, leaving {n_final:,} requests "
                       f"in {n_district_months} district-months. Counts are raw, not per resident."),
    "footer_read": (f"Hover over any point to see the exact month and request count. Every panel uses the same "
                    f"vertical scale (0 to {y_max} requests), so district heights are directly comparable."),
}


def fill_html(text, key, value):
    pattern = re.compile(r"(<!--fill:" + key + r"-->)(.*?)(<!--/fill:" + key + r"-->)", re.S)
    assert len(pattern.findall(text)) == 1, key
    return pattern.sub(lambda m: m.group(1) + escape(value, quote=False) + m.group(3), text)


def fill_js(text, key, value):
    pattern = re.compile(r"(/\*fill:" + key + r"\*/)(.*?)(/\*/fill:" + key + r"\*/)", re.S)
    assert len(pattern.findall(text)) == 1, key
    return pattern.sub(lambda m: m.group(1) + value + m.group(3), text)


assert set(re.findall(r"<!--fill:([a-z0-9_]+)-->", page)) == set(fills)
for key, value in fills.items():
    page = fill_html(page, key, value)
page = fill_js(page, "data", data_js)
page_path.write_text(page, encoding="utf-8")
for key, value in fills.items():
    print(f"{key}: {value}")

## 8. Page check

Reads the saved page back and confirms that the embedded `rows` equal `monthly_counts_by_district.csv` exactly, the y-axis ticks are round and evenly spaced, every marked text spot is filled, and the page has no em dash and no old badge.

In [ ]:
saved = page_path.read_text(encoding="utf-8")
page_rows = pd.DataFrame(json.loads(re.search(r"const rows = (\[.*?\]);", saved, re.S).group(1)))
page_meta = json.loads(re.search(r"const meta = (\{.*?\});\n", saved, re.S).group(1))
pd.testing.assert_frame_equal(page_rows, pd.read_csv(OUT / "monthly_counts_by_district.csv"))

filled = dict(re.findall(r"<!--fill:([a-z0-9_]+)-->(.*?)<!--/fill:\1-->", saved, re.S))
assert set(filled) == set(fills) and all(v.strip() for v in filled.values())
assert page_meta["y_ticks"] == list(range(0, page_meta["y_max"] + 1, Y_STEP))
assert page_meta["y_max"] >= page_rows["n"].max()
assert list(page_meta["panels"]) == DISTRICTS
assert "\u2014" not in saved and "brand-badge" not in saved and ">G1<" not in saved

print(f"page rows: {len(page_rows)}, sum {int(page_rows['n'].sum()):,}, identical to the CSV")
print("page y ticks:", page_meta["y_ticks"])
print(f"filled text spots: {len(filled)}")